# G1b · Seam 1 diagnostics, corrected and pre-registered (checklist 1.5b)

G1 (v1) said FAIL, but three of its tests could not answer its question:

1. **Part B looked in the wrong direction.** It ranked image centroids for each mapped *name*. The OOD score does
   the opposite: for each *image* it compares all prototypes. In the name → image direction, a shift shared by
   every name (the gap shift, and much of what the maps learn) favours "generic" centroids for every query.
   The score never sees that shift: for one image it adds the same amount to every prototype and cancels.
2. **Part D used the text temperature (100) for image-space similarities**, whose spread is several times larger,
   so the image score saturated at 0 or 1 and the fusion fell back to the text score. It also asked for a
   near-OOD gain, which no map can give while the negatives are NegLabel's far words (that is Seam 2's job).
3. **Part E only re-scored text**, because Part B had chosen raw text.

G1b fixes these and adds the question that decides Plan B. Every rule is fixed in the settings cell below
**before** you run it.

| Part | Question | Labels / OOD data |
|---|---|---|
| B′ | Held-out ImageNet subtrees: do mapped names pick out their own images (image → names), catch them against the ID prototypes, and separate them as pseudo-OOD? Raw and centred geometry. | ImageNet val labels judge; no OOD data |
| H | Does a map hamper CLIP? ImageNet val top-1 with mapped prototypes. | ImageNet val labels |
| D′ | OpenOOD v1.5 scores, temperature and fusion weight tuned on OpenOOD's validation split. | evaluation only |
| O | **Oracle near negatives** (diagnostic only): SSB-hard and NINCO class names added as negatives, as text, as mapped prototypes, and as the classes' real image centroids (a perfect transport). | evaluation only |
| E′ | Cold start, only if D′ keeps a map. | evaluation only |

**Runtime:** T4 GPU, about 5–10 minutes. **Output:** `MyDrive/ReNeg/oodlab_working/results/G1b_transport_diagnostics/`
— `g1b_summary.txt` (every number in plain text), `g1b_results.json` and one CSV per table.

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("G1b_transport_diagnostics", need_gpu=False)
log = ctx.log
import json, time
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
from reneg.g1b import G1b, G1bConfig
from reneg.g1 import clip_text_encoder
from reneg.wordnet import get_wordnet
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

BACKBONE = "ViT-B/16"
cache = find_cache(ctx, BACKBONE)
bank = cache.load_textbank()
wn = get_wordnet()
print("device:", ctx.device, "| WordNet:", "ready" if wn is not None else "unavailable (text-cluster groups; SSB-hard names skipped)")

### Settings (pre-registered: do not change after seeing results)

The decision rules: a map is kept only if, on the OpenOOD v1.5 test sets, it improves the best text score by
≥ 1.0 FPR95 point on near- or far-OOD **and** loses ≤ 0.5 on the other; **and** keeps ImageNet val top-1
within 1.0 point of raw text; **and** its held-out pseudo-OOD AUROC is within 0.5 of raw text's. Plan B (a
learned prior) is justified only if the real image centroids of the oracle classes beat their mapped names by
≥ 3 near FPR95 points **and** beat the text names by ≥ 3 points.

In [ ]:
cfg = G1bConfig(
    backbone=BACKBONE,
    device=ctx.device,
    p_min=0.5, n_min=3, n_cap=20,
    gain=1.0, harm=0.5, hamper=1.0, bprime_margin=0.5, planb_margin=3.0,
)
g = G1b(cache, bank, cfg, logger=log)
print(json.dumps({k: v for k, v in cfg.__dict__.items()}, indent=1, default=str))

## Preparation

The 45,000 ImageNet test images are split by index parity: half A gives the confident (pseudo-labelled) class
statistics, the mean image and the maps; half B is the ID side of every test score.

In [ ]:
prep = g.prepare()
print(json.dumps(prep, indent=1))

## B′ · Held-out subtrees, scored image → names

Columns: `name_top1` = a held-out image's nearest held-out name is its own (%); `catch` = its own name beats every
ID prototype (%); `steal` = an ID image's nearest held-out name beats its own class (%, lower is better);
`pood_*` = held-out subtrees as OOD, their names plus 3,000 NegLabel negatives as negatives (best temperature);
`t2c_top1`, `align` = G1's old direction, for comparison.

In [ ]:
b = g.part_b(wn)
print(g.tables["bprime"].round(2).to_string(index=False))

## H · Does a map hamper CLIP?

In [ ]:
h = g.part_h()
print(g.tables["hamper"].round(2).to_string(index=False))

## D′ · OpenOOD v1.5 static scores (tuned on the validation split)

`text (Z)` rows: raw text and centred text, each at its tuned temperature; `NegLabel form`: raw text at CLIP's
temperature (G1's baseline). `map` rows: the three best maps from B′ plus the gap shift, with transported ID
names (`T`) or ID centroids (`C`), alone or fused (log-odds) with the better text row.

In [ ]:
d = g.part_d()
cols = ["config", "role", "mode", "tau", "omega", "val_fpr95", "near_fpr95", "far_fpr95", "four_fpr95", "near_auroc", "far_auroc"]
print(g.tables["dprime"][cols].round(2).to_string(index=False))

## O · Oracle near negatives (diagnostic only, never a result)

The class names of SSB-hard and NINCO are added to the negatives. Half of each class's images builds its real
centroid; the other half is scored. `catch` = the image's nearest prototype is its own class (%); `id_stolen` =
an ID image's nearest prototype is one of the added classes (%).

In [ ]:
from oodlab.clipwrap import load_clip
WEIGHTS = f"{DRIVE_ROOT}/weights"
model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS)
enc = clip_text_encoder(model, prompt=cfg.prompt)
o = g.part_o(enc, wn)
print(g.tables["oracle"].round(2).to_string(index=False))

## Decision, cold start, report

In [ ]:
dec = g.decide()
e = g.part_e()
path = g.save(ctx.out)
finish(ctx, {"verdict": dec["verdict"]})
print(json.dumps({k: v for k, v in dec.items()}, indent=1, default=str))

## Summary

One plain-text block with every number above (also saved as `g1b_summary.txt`).

In [ ]:
print(g.summary_text())

## Optional · analysis pack for the local analysis

Two files of about 20 MB each with the text embeddings, class statistics, the validation split and random
subsets of the test sets (fp16, like the cache), so ideas can be tested on the real features on a CPU
(experiments/ in the repository; the Seam 1 analyses need it). They are written to
`MyDrive/ReNeg/transfers/analysis_pack/`.

In [ ]:
EXPORT_PACK = True
if EXPORT_PACK:
    for p in g.export_pack(f"{DRIVE_ROOT}/transfers/analysis_pack"):
        print(p, f"{os.path.getsize(p) / 1e6:.1f} MB")

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")